In [16]:
import pandas as pd
import psycopg2
from psycopg2 import sql


In [17]:
# Load bank information
df_banks = pd.read_csv("../data/processed/bank_reviews_cleaned.csv")

# Load review information
df_reviews = pd.read_csv("../data/processed/reviews_for_database.csv")

# Quick check
print(df_banks.head())
print(df_reviews.head())


                                              review  rating        date  \
0                                          very good       5  2025-11-25   
1           most of the time is not working properly       1  2025-11-25   
2                                       good service       5  2025-11-25   
3                                     not use for me       3  2025-11-23   
4  It keeps notifying me to disable developer opt...       1  2025-11-22   

                bank       source                             review_id  \
0  Bank of Abyssinia  Google Play  7ef21cf6-d226-4370-ab96-01c909dbc58d   
1  Bank of Abyssinia  Google Play  896ee9aa-a483-4b1f-b73c-0a26c4b54790   
2  Bank of Abyssinia  Google Play  15c3586b-e672-48db-b3c0-09508375763f   
3  Bank of Abyssinia  Google Play  6f7113d8-180e-4f3d-83d9-fbe55f9edd69   
4  Bank of Abyssinia  Google Play  d36fbdcb-b57e-4384-8e5b-ae549e25b33e   

                                         review_text  review_year  \
0                      

Connect to PostgreSQL

In [20]:
# Connect to default 'postgres' database to create our new DB
conn = psycopg2.connect(
    host="localhost",
    database="postgres",  # default DB
    user="postgres"       # your PostgreSQL user
    # password=None if no password
)

cur = conn.cursor()


Create Database

In [22]:
import psycopg2

# Connect to default 'postgres' DB
conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres"
)

# Enable autocommit
conn.autocommit = True

cur = conn.cursor()

# Drop old DB if exists, then create new
cur.execute("DROP DATABASE IF EXISTS bank_reviews;")
cur.execute("CREATE DATABASE bank_reviews;")

cur.close()
conn.close()

print("Database created successfully!")


Database created successfully!


Connect to the new database

In [23]:
import psycopg2

# Connect to the newly created database
conn = psycopg2.connect(
    host="localhost",
    database="bank_reviews",
    user="postgres"
)

cur = conn.cursor()
print("Connected to bank_reviews database")


Connected to bank_reviews database


Create the tables

In [24]:
# Create 'banks' table
cur.execute("""
CREATE TABLE IF NOT EXISTS banks (
    bank_id SERIAL PRIMARY KEY,
    bank_name VARCHAR(100) NOT NULL,
    app_name VARCHAR(100)
);
""")

# Create 'reviews' table
cur.execute("""
CREATE TABLE IF NOT EXISTS reviews (
    review_id SERIAL PRIMARY KEY,
    bank_id INT REFERENCES banks(bank_id),
    review_text TEXT NOT NULL,
    rating NUMERIC(2,1),
    review_date DATE,
    sentiment_label VARCHAR(10),
    sentiment_score NUMERIC(3,2),
    source VARCHAR(50)
);
""")

# Commit table creation
conn.commit()
print("Tables created successfully!")


Tables created successfully!


Load the processed CSV files

In [32]:
import pandas as pd

# Load processed CSVs
df_banks = pd.read_csv("../data/processed/bank_reviews_cleaned.csv")  # banks info
df_reviews = pd.read_csv("../data/processed/reviews_for_database.csv") # reviews

# Inspect the columns
print(df_banks.columns)
print(df_reviews.columns)

Index(['review', 'rating', 'date', 'bank', 'source', 'review_id',
       'review_text', 'review_year', 'review_month', 'bank_code', 'user_name',
       'thumbs_up', 'text_length', 'word_count', 'app_version'],
      dtype='object')
Index(['review_id', 'review_text', 'sentiment_label', 'sentiment_score',
       'identified_theme(s)'],
      dtype='object')


Review the data frame

In [33]:
print(df_banks.columns)
print(df_reviews.columns)

Index(['review', 'rating', 'date', 'bank', 'source', 'review_id',
       'review_text', 'review_year', 'review_month', 'bank_code', 'user_name',
       'thumbs_up', 'text_length', 'word_count', 'app_version'],
      dtype='object')
Index(['review_id', 'review_text', 'sentiment_label', 'sentiment_score',
       'identified_theme(s)'],
      dtype='object')


In [ ]:
Prepare banks DataFrame

In [35]:
# Keep only necessary columns for the banks table
df_banks_db = df_banks[['bank_code', 'bank']].drop_duplicates()

# Rename columns to match your PostgreSQL table
df_banks_db = df_banks_db.rename(columns={
    'bank_code': 'bank_id',
    'bank': 'bank_name'
})

# Optional: add app_name column if needed (or fill with None)
df_banks_db['app_name'] = None

print(df_banks_db.head())


    bank_id                    bank_name app_name
0       BOA            Bank of Abyssinia     None
400     CBE  Commercial Bank of Ethiopia     None
800  DASHEN                  Dashen Bank     None


Insert bank data

In [5]:
import pandas as pd

df_banks = pd.DataFrame([
    {"bank_code": 1, "bank": "Commercial Bank", "app_name": "CB App"},
    {"bank_code": 2, "bank": "Dashen Bank", "app_name": "Dashen App"},
    {"bank_code": 3, "bank": "Awash Bank", "app_name": "Awash App"}
])


reload the csv 